In [4]:
# 2개의 층
from tensorflow import keras
(train_input, train_target), (test_input, test_target) =\
    keras.datasets.fashion_mnist.load_data()

In [5]:
from sklearn.model_selection import train_test_split

train_scaled = train_input/255.0
train_scaled = train_scaled.reshape(-1, 28*28)
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

1절에서 만든 신경망 모델과 다른 점은 입력층과 출력층 사이에 밀집층이 추가된 것이다.

입력층과 출력층 사이에 있는 모든 층을 은닉층(Hidden Layer)이라고 한다.

출력층에 적용하는 활성화 함수는 제한적이다.
1. 이진 분류의 경우 시그모이드 함수
2. 다중 분류의 경우 소프트 맥수 함수

이에 비해 은닉층의 활성화 함수는 비교적 자유롭다.

대표적으로 시그모이드 함수와 렐루 함수 등을 사용한다.

In [6]:
# 케라스에서 신경망의 첫 번쨰 층은 input_shape 매개변수로 입력의 크기를 꼭 지정해주어야 한다.

dense1 = keras.layers.Dense(100, activation='sigmoid', input_shape=(784,))
dense2 = keras.layers.Dense(10, activation='softmax')

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# 심층 신경망 만들기

model = keras.Sequential([dense1, dense2])

💡 출력층을 가장 마지막에 두어야 한다는 것 주의

In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 100)                 │          78,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,010 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

층을 만들 때 name 매개변수로 이름 지정 가능.

출력 크기를 보면 (None, 100)이다. 첫 번째 차원은 샘플의 개수를 나타낸다.

fit() 메서드에 훈련 데이터를 주입하면 이 데이터를 한번에 모두 사용하지 않고 잘게 나누어 여러번에 걸쳐 경사 하강법 단계를 수행한다.(**미니배치 경사 하강법**)

케라스의 기본 미니배치 크기는 32개. batch_size 매개변수를 이용하여 바꿀 수 있다.

모델 파라미터 개수 계산
1. 784 x 100개 + 100개 = 78,500개
2. 100 x 10개 + 10개 = 1,010개
3. 총 79,510개의 파라미터

In [10]:
# 층을 추가하는 다른 방법

model = keras.Sequential([
    keras.layers.Dense(100, activation='sigmoid', input_shape=(784,), name='hidden'),
    keras.layers.Dense(10, activation='softmax', name='output')
    ], name='패션 MNIST 모델')

In [12]:
model.summary()

Model: "패션 MNIST 모델"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ hidden (Dense)                       │ (None, 100)                 │          78,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output (Dense)                       │ (None, 10)                  │           1,010 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model = keras.Sequential()
model.add(keras.layers.Dense(100, activation='sigmoid', input_shape=(784,), name='hidden'))
model.add(keras.layers.Dense(10, activation='softmax', name='output'))

In [14]:
# 모델 훈련

model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7535 - loss: 0.7702
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8485 - loss: 0.4173
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8624 - loss: 0.3794
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8726 - loss: 0.3548
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.8750 - loss: 0.3438


### 렐루 함수

시그모이드 함수의 경우 오른쪽과 왼쪽 끝으로 갈수록 그래프가 누워있기 때문에 올바른 출력을 만드는데 신속하게 대응하지 못한다.

**렐루함수**

$\phi = \frac{1}{1+\exp^{-z}}$

입력이 양수일 경우 마치 활성화 함수가 없는 것처럼 입력을 통과시키고 음수일 경우 0으로 만든다.

렐루 함수는 `max(0,z)`와 같이 쓸 수 있다.

이미지 처리에서 좋은 성능을 낸다고 알려져 있다.

In [15]:
model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(28, 28)))
model.add(keras.layers.Dense(100, activation='relu'))
model.add(keras.layers.Dense(10, activation='softmax'))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Flatten 클래스는 배치 차원을 제외하고 나머지 입력 차원을 모두 일렬로 펼치는 역할만 한다.

인공 신경망의 성능을 위해 기여하는 바는 없다.

Flatten 층은 입력층 바로 뒤에 추가한다.

신경망을 깊이가 3인 신경망이라고 부르지는 않는다.

In [16]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 100)                 │          78,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │           1,010 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

Flatten 클래스에 포함된 모델 파라미터 개수는 0개이다.

케라스의 Flatten층을 신경망 모델에 추가하면 입력값의 차원을 짐작할 수 있는 것이 장점이다.



In [18]:
(train_input, train_target), (test_input, test_target) =\
    keras.datasets.fashion_mnist.load_data()


train_scaled = train_input / 255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [19]:
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7637 - loss: 0.6863
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8516 - loss: 0.4103
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8722 - loss: 0.3521
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8813 - loss: 0.3293
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8891 - loss: 0.3100


시그모이드 함수를 사용했을 때와 비교하면 성능이 조금 향상

accuracy

- sigmoid : 0.8750

- relu : 0.8891


In [21]:
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8846 - loss: 0.3374


[0.3430631160736084, 0.8829166889190674]

### 옵티마이저(Optimizer)
가장 기본적인 Optimizer는 확률적 경사 하강법인 SGD이다.

SGD Optimizer를 사용하려면 compile() 메서드의 optimizer 매개변수를 'sgd'로 지정한다.

```
model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy',metrics=['accurcay'])
```

```
sgd = keras.optimizers.SGD()
model.compile(optimizer=sgd, loss='sparse_categorical_crossentropy',metrics=['accurcay'])
```

sgd와 'sgd'의 차이점


> 원래는 sgd = keras.optimizers.SGD()처럼 클래스 객체를 만들어 사용해야 하지만 'sgd'로 지정하면 자동으로 SGD 클래스 객체를 만들어 준다.

SGD 클래스의 학습률 기본값이 0.01일 때 이를 바꾸고 싶다면 learning_rate 매개변수에 지정하여 사용한다.

```
sgd = keras.optimizers.SGD(learning_rate=0.1)
```


기본 경사 하강법 optimizer는 모두 SGD 클래스에서 제공한다. SGD 클래스의 momentum 매개변수의 기본값은 0이다.

이를 0보다 큰 값으로 지정하면 마치 이전의 그래디언트를 가속도처럼 사용하는 **모멘텀 최적화(Momentum Optimization)**를 사용한다.

보통 momentum 매개변수는 0.9 이상을 지정한다.

SGD 클래스의 nesterov 매개변수를 기본값 False에서 True로 바꾸면 네스테로프 모멘텀 최적화(Nestrov Momentum Optimization)를 사용한다.

```
sgd = keras.optimizers.SGD(momentum=0.9, nestrov=True)
```
네스테로프 모멘텀은 모멘텀 최적화를 2번 반복하여 구현한다. 대부분 기본 확률적 경사 하강법보다 더 나은 성능을 제공한다.

모델이 최적점에 가까이 갈수록 학습률을 낮출 수 있다. 안정적으로 최적점에 수렴할 가능성이 높다.

이런 학습률을 적응적 학습률(Adaptive Learning Rate)이라고 한다.

대표적인 옵티마이저는 Adagrad와 RMSprop이다. optimizer 매개변수의 기본값은 'rmsprop'이다.

```
adagrad = keras.optimizers.Adagrad()
model.compile(optimizer=adagrad, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
```

RMSprop도 동일

```
rmsprop = keras.optimizers.RMSprop()
model.compile(optimizer=rmsprop, loss='sparse_categorical_crossentropy', mertics=['accuracy'])
```
모멘텀 최적화와 RMSprop의 장점을 접목한 것이 Adam이다.

In [22]:
# Adam 클래스의 매개변수 기본값을 사용해 패션 MNIST 모델 훈련
model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(28, 28)))
model.add(keras.layers.Dense(100, activation='relu'))
model.add(keras.layers.Dense(10, activation='softmax'))

In [23]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7697 - loss: 0.6680
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8582 - loss: 0.4041
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.8687 - loss: 0.3615
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8807 - loss: 0.3274
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8881 - loss: 0.3062


In [24]:
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8771 - loss: 0.3441


[0.34160318970680237, 0.8743333220481873]